<a href="https://colab.research.google.com/github/Denniskag/Python-Project-for-Data-Science/blob/main/Explainable_Churn_Prediction_with_SHAP_%26_LIME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
rhythmghai_250k_customer_churn_prediction_dataset_path = kagglehub.dataset_download('rhythmghai/250k-customer-churn-prediction-dataset')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

**Importing dataset**

The dataset consists of 250,000 synthetic telecom customer records, generated to closely mimic real world customer behavior patterns.

In [ ]:
df=pd.read_csv('/kaggle/input/datasets/rhythmghai/250k-customer-churn-prediction-dataset/customer_churn_dataset_250k.csv')

In [ ]:
df.head()

In [ ]:
df.info()

**Checking for null values**

In [ ]:
df.isnull().sum()

**Handling missing values in the "internet_service" column by replacing them with the mode**

In [ ]:
mode_value = df['internet_service'].mode()[0]

df['internet_service'] = df['internet_service'].fillna(mode_value)

In [ ]:
df.isnull().sum()

**Exploring the distribution of the target variable "churn" to assess class balance**

In [ ]:
sns.countplot(data=df,x="churn")
plt.show()

In [ ]:
# Returns a list of column names
object_columns = df.select_dtypes(include=['object']).columns.tolist()
print(object_columns)

**Endoding**

Nominal-['region', 'internet_service', 'payment_method']-use get dummies

Ordinal-['contract_type']- use label/binary encoding

Gender-Binary encoding

In [ ]:
for_dummies=['region', 'internet_service', 'payment_method']

for_le_encoding=['gender','contract_type']

In [ ]:
df = pd.get_dummies(df, columns=['region', 'internet_service', 'payment_method'],dtype=int)

In [ ]:
df['gender']=df['gender'].map({'Male':0,'Female':1})

In [ ]:
df['contract_type']=df['contract_type'].map({'Month-to-Month': 0, 'One Year': 1, 'Two Year': 2})

In [ ]:
df.head()

**Training the model**

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

X = df.drop(["customer_id", "churn"], axis=1)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = xgb.XGBClassifier()
model.fit(X_train, y_train)

**Faeture Importance and Explainability using SHAP and LIME**

In [ ]:
import shap

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.waterfall(shap_values[0])

**Lime**

In [ ]:
import lime
import lime.lime_tabular
import numpy as np

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X_train.columns,
    class_names=['Not Churn', 'Churn'],
    mode='classification'
)

In [ ]:
i = 500
exp = explainer.explain_instance(
    data_row=np.array(X_test.iloc[i]),
    predict_fn=model.predict_proba
)

In [ ]:
exp.show_in_notebook(show_table=True)

In [ ]:
fig = exp.as_pyplot_figure()
fig.set_size_inches(10,6)
plt.show()